# US Flight Delays and Cancellations: Prediction Report_2

**Author:** Stephen Handy
**Course:** CSC381-01
**Professor:** Yunlong Wang
**Term:** Fall 2026

## Abstract

This report investigates whether 2025-2026 flight data can be used to predict flight delays or cancellations before departure. Analysis includes data, preprocessing decisions, models, evaluation results, and limitations.

## 1. Background

Flight delays and cancellations affect passengers, airlines, airports, and connecting flights. A pre-departure prediction can help passengers assess travel risk and airlines & airports to plan staffing and operations. 

False negatives may cause travelers to miss connections or arrive unprepared. False positives may create unnecessary concern or operational cost. 

## Research Question

Can a flight's pre-departure information predict whether this flight will be delayed or cancelled?

## Hypothesis

Airline, route, airport, schedule departure time, and date features will provide predictive information. However, weather, maintenance, air-traffic conditions, and unexpected operational events will limit predictive accuracy. 

The target variable, 'disrupted', equals 1 when a flight is delayed or cancelled and 0 otherwise.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 381


All models use the same random seed and test set so that their results can be compared fairly. 

## 3. Dataset

The dataset contains US flight records from 2025-2026.
Each row represents one scheduled flight.

**Source:** https://www.kaggle.com/datasets/samuelpombourcq/us-flight-delays-and-cancellations-2025-2026
**Retrieved:** November 2026
**Target:** Whether the flight was delayed or cancelled.

In [ ]:
DATA_PATH = "path/to/flight_data.csv"

df = pd.read_csv(DATA_PATH)

display(df.head())
print("Rows and columns:", df.shape)
display(df.dtypes)


(500000, 40)


,flight_date,year,month,day_of_week,airline_code,airline,tail_number,flight_number,origin,origin_city,...,diverted,sched_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2025-01-01,2025,1,3,AA,American Airlines,N960NN,2158,SMF,"Sacramento, CA",...,0,249,221.0,202.0,1781,NaN,NaN,NaN,NaN,NaN
1,2025-01-01,2025,1,3,B6,JetBlue Airways,N3145J,98,DEN,"Denver, CO",...,0,226,215.0,180.0,1626,0.0,0.0,0.0,0.0,96.0
2,2025-01-01,2025,1,3,AS,Alaska Airlines,N973AK,84,ANC,"Anchorage, AK",...,0,213,192.0,176.0,1448,NaN,NaN,NaN,NaN,NaN
3,2025-01-01,2025,1,3,AS,Alaska Airlines,N296AK,704,SEA,"Seattle, WA",...,0,152,128.0,110.0,867,NaN,NaN,NaN,NaN,NaN
4,2025-01-01,2025,1,3,B6,JetBlue Airways,N955JB,2169,BOS,"Boston, MA",...,0,212,204.0,184.0,1237,NaN,NaN,NaN,NaN,NaN


In [ ]:
print("Duplicate rows:", df.duplicated().sum())

missing_values = (
    df.isna()
        .sum()
        .sort_values(ascending=False)
)

display(missing_values[missing_values > 0])


In [2]:
df = df[
    (df["cancelled"] == 0)
    & (df["diverted"] == 0)
    & (df["arr_delay"].notna())
].copy()


In [3]:
df["sched_dep_hour"] = df["sched_dep_time"] // 100


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

feature_columns = [
    "airline_code", "origin", "dest", "month", "day_of_week", "sched_dep_hour", "dep_time_block", "distance", "sched_elapsed_time"
]

categorical = [
    "airline_code", "origin", "dest", "month", "day_of_week", "dep_time_block"
]

numeric = ["sched_dep_hour", "distance", "sched_elapsed_time"]

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("numeric", "passthrough", numeric)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0))
])

df["flight_date"] = pd.to_datetime(df["flight_date"])
df = df.sort_values("flight_date")

split_date = df["flight_date"].quantile(0.80)

train = df[df["flight_date"] <= split_date].copy()
test = df[df["flight_date"] > split_date].copy()

X_train = train[feature_columns]
y_train = train["arr_delay"]

X_test = test[feature_columns]
y_test = test["arr_delay"]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)


Training rows: 392438
Testing rows: 97198
MAE: 31.655815190220064
RMSE: 64.86979731749867
R²: 0.029068151843322787
